# LLM 推理为什么慢：Prefill、Decode 与 KV Cache

> 上一章解决了 `logits → next token`。但每产生一个新的 Token，都必须重新调用一次 Transformer Forward。
>
> 本章只回答一个问题：**一次 Token 生成为什么慢，慢在哪里？**
>
> 我们沿着推理链逐层拆：
>
> 1. **Prefill 与 Decode**：为什么同一个模型会有两种完全不同的计算阶段？
> 2. **KV Cache**：为什么历史 Token 不应该每一步都重算？
> 3. **MHA / GQA / MQA**：为什么很多新模型在减少 KV Head？
> 4. **TTFT / TPOT**：为什么“首 Token 慢”和“后续 Token 慢”不是同一个问题？
> 5. **Memory-bound Decode**：为什么大模型推理常常不是算力不够，而是权重搬得太慢？
>
> 这一章先把单请求的计算路径看清。后面的量化、投机解码、Serving System 都是在这张图上继续做优化。

先看一个最简单的请求：

```text
Prompt:  "请解释一下 Transformer"
输入长度: 1000 tokens
输出长度: 500 tokens
```

这 1500 个 Token 并不是用同一种方式计算的。

## 1. Prefill 和 Decode：同一次请求里的两种世界

处理 Prompt 时，1000 个输入 Token 已经全部知道，所以模型可以一次并行处理整段输入。

这一步叫 **Prefill**。

```text
1000-token Prompt
       ↓
一次大矩阵计算
       ↓
得到每一层的 KV Cache
       ↓
得到第一个输出位置的 logits
```

从第一个输出 Token 开始，情况变了。

第二个输出 Token 依赖第一个输出 Token，第三个又依赖前两个。因此生成阶段必须一步一步走。

这一步叫 **Decode**。

```text
Token 1 → Forward → Token 2
Token 2 → Forward → Token 3
Token 3 → Forward → Token 4
...
```

所以：

> **Prefill 处理一整段已知 Prompt；Decode 每次只新增一个 Token。**

In [ ]:
prompt_len = 1000
output_len = 500

print("Prefill 处理 token 数:", prompt_len)
print("Decode 需要 forward 次数:", output_len)
print("如果输出 500 token，至少要经历 500 次串行 decode step。")

这个区别非常重要，因为后面几乎所有推理系统术语都在问：

> 它优化的是 Prefill，还是 Decode？

例如：

- FlashAttention 对长 Prompt 的 Prefill 非常重要。
- KV Cache 直接改变 Decode 的历史状态复用。
- 量化常常能显著降低 Decode 的权重带宽压力。
- PD 分离则干脆把 Prefill 和 Decode 放到不同 Worker 上。

但在看这些优化前，先看最朴素的 Decode 到底浪费在哪里。

## 2. 没有 KV Cache：历史 Token 为什么会被一遍遍重算？

假设已经生成：

```text
[BOS, 我, 喜欢, 机器]
```

如果每一步都把完整前缀重新送进模型：

```text
Step 1: [BOS]
Step 2: [BOS, 我]
Step 3: [BOS, 我, 喜欢]
Step 4: [BOS, 我, 喜欢, 机器]
```

那么 `BOS` 的 K、V 在每一步都会重新算。

`我` 在 Step 2 算过一次，Step 3、Step 4 又继续重算。

这就是最直接的重复计算。

In [ ]:
def processed_prefix_tokens(n):
    return sum(range(1, n + 1))

for n in [10, 100, 1000]:
    naive = processed_prefix_tokens(n)
    print(f"生成 {n:4d} 个 token，朴素前缀处理代理量 = {naive:,}")

这里的 `1 + 2 + ... + N` 只是一个**教学代理量**，用来说明“历史前缀被反复处理”的增长速度。

不要把它机械记成：

```text
没有 KV Cache = O(N²)
有 KV Cache = O(N)
```

真实 Transformer 还包含 Attention、MLP、不同层、不同 context length 的计算。KV Cache 真正省掉的是：

> **历史 Token 的 K / V 不需要每一步重新经过投影层计算。**

新 Token 仍然要和所有历史 KV 做 Attention。

## 3. KV Cache：把已经算过的 K、V 存起来

Self-Attention 中每个 Token 都会产生：

```text
Q = XWq
K = XWk
V = XWv
```

在 Decode 时，历史 Token 的 K、V 已经不会改变。

所以可以把它们保存下来：

```text
过去 Token 的 K/V ─────┐
                       ├→ Attention
新 Token 只计算 Q/K/V ─┘
```

这块保存历史 K/V 的显存，就是 **KV Cache**。

In [ ]:
# 一个简化的 KV Cache 大小估算
def kv_cache_gb(
    num_layers=32,
    num_kv_heads=32,
    head_dim=128,
    seq_len=4096,
    batch_size=1,
    bytes_per_value=2,
):
    # K 和 V 两份
    total_bytes = (
        2 * num_layers * num_kv_heads * head_dim *
        seq_len * batch_size * bytes_per_value
    )
    return total_bytes / 1e9

for seq in [2048, 8192, 32768]:
    print(f"seq={seq:5d} -> KV Cache ≈ {kv_cache_gb(seq_len=seq):.2f} GB")

这段计算暴露了 KV Cache 的代价：

> **省了计算，却开始吃显存。**

而且它会随着这些量线性增长：

```text
batch size
× sequence length
× number of layers
× number of KV heads
× head dimension
× precision bytes
```

这也是为什么服务系统里同时跑很多长请求时，KV Cache 往往比“模型权重是否能放下”更早成为容量问题。

于是下一问自然出现：

> **能不能少存一些 KV Head？**

## 4. MHA、GQA、MQA：为什么 KV Head 越来越少？

标准 Multi-Head Attention（MHA）里：

```text
Q heads = 32
K heads = 32
V heads = 32
```

每个 Query Head 都有自己的 K/V Head。

但 Decode 时，KV Cache 的大小正比于 KV Head 数。

所以后来出现了：

### MQA — Multi-Query Attention

```text
Q heads = 32
KV heads = 1
```

所有 Query Head 共享一组 K/V。

### GQA — Grouped-Query Attention

```text
Q heads = 32
KV heads = 8
```

每 4 个 Query Head 共享一组 K/V。

GQA 是 MHA 和 MQA 之间的折中。今天很多 LLM 都采用 GQA。

In [ ]:
layers = 32
head_dim = 128
seq_len = 8192

for kv_heads in [32, 8, 1]:
    size = kv_cache_gb(
        num_layers=layers,
        num_kv_heads=kv_heads,
        head_dim=head_dim,
        seq_len=seq_len,
    )
    label = {32:"MHA", 8:"GQA", 1:"MQA"}[kv_heads]
    print(f"{label:<4} KV heads={kv_heads:2d} -> KV Cache ≈ {size:.2f} GB")

看到这里，再读模型 Config 里的：

```text
num_attention_heads = 32
num_key_value_heads = 8
```

就不应该只把它当参数表。

它直接意味着：

> 这是一个 GQA 模型，KV Cache 比 32 个 KV Head 的 MHA 更小。

这就是“看懂模型结构参数”和“看懂推理系统”开始接上的地方。

## 5. TTFT 和 TPOT：为什么用户会感受到两种不同的慢？

现在我们已经知道：

```text
Prompt → Prefill → 第一个 Token
后续 Token → Decode → Decode → Decode...
```

服务系统通常把这两种等待拆成不同指标。

### TTFT — Time To First Token

从请求到达，到用户看到第一个输出 Token。

它包含：

- 排队时间
- Prompt Tokenization
- Prefill
- 调度等待
- 第一个 Decode

长 Prompt 往往会明显拉高 TTFT。

### TPOT — Time Per Output Token

第一个 Token 之后，每个输出 Token 之间的平均时间。

它更接近 Decode 性能。

所以：

```text
首字等很久，但后面刷得快 → TTFT 差
很快开始输出，但一个字一个字蹦得慢 → TPOT 差
```

In [ ]:
# 用一组 toy latency 感受 TTFT / TPOT
request_arrival = 0.0
first_token_time = 0.85
token_times = [0.85, 0.90, 0.95, 1.01, 1.06]

ttft = first_token_time - request_arrival
tpot = sum(
    token_times[i] - token_times[i-1]
    for i in range(1, len(token_times))
) / (len(token_times)-1)

print(f"TTFT = {ttft:.3f} s")
print(f"TPOT = {tpot:.3f} s/token")

以后看大厂推理报告时，一定要警惕一句模糊的话：

> “Latency 降低了 30%。”

应该继续问：

- 是 TTFT 降了？
- 还是 TPOT 降了？
- P50 还是 P99？
- 什么 Prompt length？
- 什么 output length？
- 什么 concurrency？

如果这些条件不说，单独一个“latency”数字几乎无法比较。

## 6. 为什么 Decode 常常是 Memory-bound？

一个 7B 模型如果用 BF16 权重：

```text
7B parameters × 2 bytes ≈ 14 GB
```

Decode 每一步只新增一个 Token，矩阵计算规模相对小，但模型权重仍然要被读取。

于是会出现一个很重要的直觉：

> GPU 算力可能很强，但如果每一步都要搬很多 GB 的权重，瓶颈可能先出现在显存带宽，而不是 FLOPs。

这类问题常用 **Arithmetic Intensity / Roofline Model** 来理解。

In [ ]:
params = 7e9
bytes_per_param = 2
weight_bytes = params * bytes_per_param

gpu_bw = 2_000e9  # 2 TB/s 的教学近似
ideal_weight_reads_per_sec = gpu_bw / weight_bytes

print(f"7B BF16 权重 ≈ {weight_bytes/1e9:.1f} GB")
print(f"若每个 decode step 都近似读一遍权重")
print(f"仅按带宽上限估算 ≈ {ideal_weight_reads_per_sec:.1f} 次/s")
print("真实系统会受 batching、KV、kernel、cache 等因素影响。")

这段估算不是性能预测，只是为了建立一个方向：

```text
Decode:
每次计算量不算大
但权重很大
→ arithmetic intensity 低
→ 容易 memory-bound
```

这也解释了下一章为什么自然是 **Quantization**。

如果 BF16 权重 14 GB，而 INT4 理论上只有约 3.5 GB，那么每一步需要搬的数据会明显减少。

## 7. FlashAttention 放在哪里？

看到这里很多人会问：

> FlashAttention 不是最经典的推理加速技术之一吗？

是，但它解决的是另一个层面的瓶颈。

朴素 Attention 会生成很大的中间矩阵，并频繁读写 HBM。FlashAttention 通过 **tiling / IO-aware** 的方式减少中间结果的显存读写。

在长 Prompt 的 Prefill 阶段，这尤其重要。

先记住位置：

```text
KV Cache      → 不重复算历史 K/V
GQA / MQA     → 减少 KV Cache 大小
FlashAttention→ 减少 Attention 中间数据搬运
Quantization  → 减少权重 / Activation / KV 的位宽
```

后面 24 章会把 FlashAttention / FlashInfer 放回完整 Serving Stack。

## 8. 一张图串起来

```text
请求到达
   ↓
Prompt Tokens
   ↓
Prefill ─────────────→ TTFT
   │
   ├─ 计算所有 Prompt K/V
   └─ 写入 KV Cache
   ↓
Decode Step 1
   ↓
Decode Step 2
   ↓
Decode Step 3 ───────→ TPOT
   ...
```

Decode 每一步：

```text
读模型权重
+ 读历史 KV Cache
+ 计算新 Token
+ 写新的 K/V
```

这就是后续所有优化的出发点。

## 小结

- **Prefill**：并行处理完整 Prompt，通常更 compute-intensive。
- **Decode**：一次生成一个 Token，严格串行，常更 memory-bound。
- **KV Cache**：保存历史 K/V，避免重复计算。
- **MHA / GQA / MQA**：KV Head 数越来越少，本质上是在压 KV Cache。
- **TTFT**：从请求到首 Token。
- **TPOT**：后续 Token 的生成间隔。
- **FlashAttention**：主要优化 Attention 的 IO。
- **Memory-bound Decode**：解释了为什么量化、batching、kernel 优化如此重要。

下一章继续追问：

> 权重本身太大、每一步搬得太慢，能不能直接把数字存得更小？

## 作业

1. 用公式估算一个 32 层、8 KV Head、head_dim=128、context=32k 的 BF16 KV Cache。
2. 把 KV Head 从 32 改成 8，KV Cache 理论缩小几倍？
3. 一个请求 TTFT=1.2s、TPOT=40ms，另一个 TTFT=0.3s、TPOT=90ms，哪个“更快”？为什么不能只给一个答案？
4. 解释为什么 KV Cache 不等于“把 Attention 从 O(N²) 变成 O(N)”。